# CarDD YOLO11 fine-tune

YOLO11 on CarDD (6 classes: dent, scratch, crack, glass shatter, lamp broken, tire flat).
Model size set via `MODEL_VARIANT` below.

Setup: attach `CarDD_COCO` as a dataset input, GPU + Internet on, run top to bottom.


In [ ]:
!pip install -q ultralytics==8.4.90

import torch, ultralytics
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Enable GPU in Notebook Settings (Accelerator: GPU T4 x2 or P100) before continuing."


## Confirm dataset mount path
Set `KAGGLE_INPUT_DIR` below to match.

In [ ]:
!ls /kaggle/input

In [ ]:
import json
from pathlib import Path

# <-- edit this to match the folder name printed above
KAGGLE_INPUT_DIR = Path("/kaggle/input/cardd-coco")

# Model size: yolo11n (nano) / yolo11s (small) / yolo11m / yolo11l / yolo11x
# Swap here to try a different size later - everything below reads from this one setting.
MODEL_VARIANT = "yolo11n"
RUN_NAME = f"cardd_{MODEL_VARIANT}"

COCO_ROOT = KAGGLE_INPUT_DIR / "CarDD_COCO" if (KAGGLE_INPUT_DIR / "CarDD_COCO").exists() else KAGGLE_INPUT_DIR
OUT_ROOT = Path("/kaggle/working/cardd_yolo")
SPLITS = ["train2017", "val2017", "test2017"]
EXPECTED_COUNTS = {"train2017": 2816, "val2017": 810, "test2017": 374}

assert (COCO_ROOT / "annotations" / "instances_train2017.json").exists(), (
    f"Can't find CarDD annotations under {COCO_ROOT} - check KAGGLE_INPUT_DIR matches the mount name above"
)

# Class names read from the dataset's own JSON rather than hardcoded (matches scripts/convert_coco_to_yolo.py)
with open(COCO_ROOT / "annotations" / "instances_train2017.json") as f:
    _categories = json.load(f)["categories"]
_ids = sorted(c["id"] for c in _categories)
assert _ids == list(range(1, len(_ids) + 1)), f"Expected contiguous category ids starting at 1, got {_ids}"
CLASS_NAMES = [c["name"] for c in sorted(_categories, key=lambda c: c["id"])]

print("Using COCO_ROOT =", COCO_ROOT)
print("Model variant =", MODEL_VARIANT, "-> run name:", RUN_NAME)
print("Class names:", CLASS_NAMES)


## COCO -> YOLO conversion
`cls91to80=False` — CarDD's category ids are custom, not COCO's.

In [ ]:
import shutil
from ultralytics.data.converter import convert_coco

if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)  # convert_coco increments the dir name instead of overwriting

convert_coco(
    labels_dir=str(COCO_ROOT / "annotations"),
    save_dir=str(OUT_ROOT),
    use_segments=False,
    use_keypoints=False,
    cls91to80=False,
)

for split in SPLITS:
    src_dir = COCO_ROOT / split
    dst_dir = OUT_ROOT / "images" / split
    dst_dir.mkdir(parents=True, exist_ok=True)
    for img_path in src_dir.glob("*.jpg"):
        dst = dst_dir / img_path.name
        if not dst.exists():
            shutil.copy2(img_path, dst)

print(f"{'split':12s}{'images':>10s}{'label_files':>14s}")
for split in SPLITS:
    n_images = len(list((OUT_ROOT / "images" / split).glob("*.jpg")))
    n_labels = len(list((OUT_ROOT / "labels" / split).glob("*.txt")))
    print(f"{split:12s}{n_images:10d}{n_labels:14d}")
    if n_images != EXPECTED_COUNTS[split]:
        print(f"  WARNING: expected {EXPECTED_COUNTS[split]} images for split '{split}', found "
              f"{n_images} - check the upload/attachment completed correctly.")


In [ ]:
import yaml

data_yaml = {
    "path": str(OUT_ROOT.resolve()),
    "train": "images/train2017",
    "val": "images/val2017",
    "test": "images/test2017",
    "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}
data_yaml_path = Path("/kaggle/working/cardd_yolo.yaml")
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)
print(data_yaml_path.read_text())


## Sanity check
Redraw decoded YOLO boxes to confirm the conversion before training.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(0)
img_dir = OUT_ROOT / "images/train2017"
lbl_dir = OUT_ROOT / "labels/train2017"
sample = random.sample(sorted(img_dir.glob("*.jpg")), 6)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, img_path in zip(axes.flatten(), sample):
    im = plt.imread(img_path)
    h, w = im.shape[:2]
    ax.imshow(im)
    lbl_path = lbl_dir / (img_path.stem + ".txt")
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            cls, cx, cy, bw, bh = line.split()[:5]
            cls = int(cls)
            cx, cy, bw, bh = float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
            x0, y0 = cx - bw / 2, cy - bh / 2
            ax.add_patch(patches.Rectangle((x0, y0), bw, bh, linewidth=1.5, edgecolor="lime", facecolor="none"))
            ax.text(x0, max(y0 - 4, 0), CLASS_NAMES[cls], color="lime", fontsize=8,
                     bbox=dict(facecolor="black", alpha=0.5, pad=0.5, edgecolor="none"))
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")
fig.tight_layout()
plt.savefig("/kaggle/working/sanity_check.png", dpi=150)
plt.show()


## Train
Fixed seed, early stopping via `patience`, RAM caching (dataset is small).

In [ ]:
from ultralytics import YOLO

model = YOLO(f"{MODEL_VARIANT}.pt")
train_results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    seed=0,
    deterministic=True,
    patience=20,
    device=0,
    cache="ram",
    project="/kaggle/working/runs",
    name=RUN_NAME,
    exist_ok=True,
)


## Test set evaluation
Separate from training-time val — this is the headline number.

In [ ]:
best_weights = f"/kaggle/working/runs/{RUN_NAME}/weights/best.pt"
test_model = YOLO(best_weights)
test_metrics = test_model.val(
    data=str(data_yaml_path),
    split="test",
    project="/kaggle/working/runs",
    name=f"{RUN_NAME}_test",
)

print("Test mAP50:", test_metrics.box.map50)
print("Test mAP50-95:", test_metrics.box.map)
print("\nPer-class mAP50-95:")
for name, ap in zip(CLASS_NAMES, test_metrics.box.maps):
    print(f"  {name:15s} {ap:.4f}")


## Qualitative examples
Sample predictions for the README.

In [ ]:
import random
random.seed(1)
test_images = sorted((OUT_ROOT / "images/test2017").glob("*.jpg"))
sample_imgs = random.sample(test_images, 8)
test_model.predict(
    source=[str(p) for p in sample_imgs],
    save=True,
    conf=0.25,
    project="/kaggle/working/runs",
    name=f"{RUN_NAME}_qualitative",
)


## Package outputs
Download the zip + this notebook into the local repo.

In [ ]:
import shutil

zip_path = f"/kaggle/working/{RUN_NAME}_results"
shutil.make_archive(zip_path, "zip", "/kaggle/working/runs")
print(f"Zipped run artifacts to {zip_path}.zip\n")
print("Key files inside runs/:")
print(f"  weights:               {RUN_NAME}/weights/best.pt")
print(f"  train metrics/plots:   {RUN_NAME}/results.csv, results.png, confusion_matrix.png, PR_curve.png")
print(f"  held-out test metrics: {RUN_NAME}_test/")
print(f"  qualitative preds:     {RUN_NAME}_qualitative/")
